CREAR LAS RESAPECTIVAS CLASES

In [26]:
class_mapping = {
    0: "aire_acondicionado",
    1: "bocina_auto",
    2: "niños_jugando",
    3: "ladrido_perro",
    4: "taladro",
    5: "motor_encendido",
    6: "disparo",
    7: "martillo_neumatico",
    8: "sirena",
    9: "musica_callejera"
}

IMPORTAMOS LIBRERIAS

In [27]:
import numpy as np
from IPython.display import Audio, display
import torch
import torch.nn.functional as F
import torchaudio.transforms as T
import random
import torch.nn as nn
from tqdm import tqdm



IMPORTAMOS LOS DATOS INDIVIDUALEMNTE 

In [28]:
X1 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold1_X.npy")
Y1 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold1_Y.npy")

X2 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold2_X.npy")
Y2 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold2_Y.npy")

X3 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold3_X.npy")
Y3 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold3_Y.npy")

X4 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold4_X.npy")
Y4 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold4_Y.npy")

X5 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold5_X.npy")
Y5 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold5_Y.npy")

X6 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold6_X.npy")
Y6 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold6_Y.npy")

X7 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold7_X.npy")
Y7 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold7_Y.npy")

X8 = np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold8_X.npy")
Y8 = np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold8_Y.npy")

X9 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold9_X.npy")
Y9 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold9_Y.npy")

X10 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold10_X.npy")
Y10 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold10_Y.npy")

In [29]:
X10 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold10_X.npy")
Y10 =np.load(r"F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS COMPLETOS\UrbanSound8K\audio\fold10_Y.npy")

In [30]:
X10.shape

(837, 88200)

PROBAMOS UN AUDIO TENIENDO EN CEUNTA QUE SAMPLE_RATE   = 22050


In [31]:
m=11
for m in range(1,100,7):
    
    display(Audio(X4[m],rate=22050))

    print(f'el sonido es de {class_mapping[Y4[m]]}')
      

el sonido es de musica_callejera


el sonido es de ladrido_perro


el sonido es de musica_callejera


el sonido es de ladrido_perro


el sonido es de motor_encendido


...

el sonido es de musica_callejera


el sonido es de disparo


el sonido es de disparo


el sonido es de motor_encendido


el sonido es de motor_encendido


ya mirando las dimesiones, y como se organzan als clases, procedemos

ahora agregando las aumentaciones 

RUIDO GAUSSIANO

SNR ALEATORIO HASTA 30dB

TIME SHIFT

MAG WARPING

POLARITY INV

RANDOM GAIN 

CLIPPING

TIME DROPOUT

LOW PASS

Organizamos en folds

In [32]:
X_folds = [X1, X2, X3, X4, X5, X6, X7, X8, X9, X10]
Y_folds = [Y1, Y2, Y3, Y4, Y5, Y6, Y7, Y8, Y9, Y10]

Funciones de Augmentation

In [58]:
def gaussian_noise(w):
    noise_level = torch.empty(1).uniform_(0.005, 0.025).item()
    return (w + torch.randn_like(w) * noise_level).clamp(-1, 1)


def snr_noise(w):
    snr_db = torch.empty(1).uniform_(15, 40).item()
    signal_power = w.pow(2).mean()
    if signal_power < 1e-8:
        return w
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = torch.randn_like(w) * torch.sqrt(noise_power)
    return (w + noise).clamp(-1, 1)


def magnitude_warping(w):
    N = w.shape[1]
    puntos = (1.0 + 0.15 * torch.randn(8)).unsqueeze(0).unsqueeze(0).to(w.device)
    curva = F.interpolate(puntos, size=N, mode="linear", align_corners=True).squeeze(0)
    return (w * curva).clamp(-1, 1)


def polarity_inversion(w):
    return w * -1


def clipping(w):
    threshold = torch.empty(1).uniform_(0.4, 0.9).item()
    return torch.clamp(w, -threshold, threshold)


def time_dropout(w):
    total = w.shape[1]
    max_dur = int(0.15 * total)
    dur = torch.randint(int(0.05 * total), max_dur, (1,)).item()
    start = torch.randint(0, total - dur, (1,)).item()
    w_out = w.clone()
    w_out[:, start:start + dur] = 0
    return w_out


def low_pass_filter(w):
    kernel_size = torch.randint(5, 25, (1,)).item()
    kernel_size = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
    return F.avg_pool1d(
        w.unsqueeze(0),
        kernel_size=kernel_size,
        stride=1,
        padding=kernel_size // 2
    ).squeeze(0)


def time_shift(w):
    sr = 22050
    max_m = int(0.5 * sr)
    shift = torch.randint(-max_m, max_m, (1,)).item()
    return torch.roll(w, shifts=shift, dims=1)


def ganancia(w):
    factor = float(torch.empty(1).uniform_(0.7, 1.4).item())
    return (w * factor).clamp(-1, 1)

In [34]:
def apply_augmentation(w, p=0.5):
    augmentations = [
        gaussian_noise,
        snr_noise,
        magnitude_warping,
        polarity_inversion,
        clipping,
        time_dropout,
        low_pass_filter,
        time_shift,
        ganancia,
    ]
    for aug in augmentations:
        if random.random() < p:
            w = aug(w)
    return w

In [59]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

mel_transform_gpu = T.MelSpectrogram(
    sample_rate=22050, n_fft=1024, hop_length=512,
    n_mels=128, f_min=0.0, f_max=11025.0,
).to(device)

amplitude_to_db_gpu = T.AmplitudeToDB(stype='power', top_db=80).to(device)

def waveform_to_mel(w):
    mel = mel_transform(w)
    mel = amplitude_to_db(mel)
    mel = (mel - mel.mean()) / (mel.std() + 1e-8)
    return mel


def waveform_to_mel(w):
    mel = mel_transform_gpu(w)
    mel = amplitude_to_db_gpu(mel)
    mel = (mel - mel.mean()) / (mel.std() + 1e-8)
    return mel
# ── Device ────────────────────────────────────────────────────────────────────

    


cuda
Usando: cuda
NVIDIA GeForce RTX 3050 Laptop GPU


In [60]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

Thu May  7 23:11:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.21                 Driver Version: 596.21         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   66C    P8              9W /   30W |    1661MiB /   4096MiB |     30%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [64]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(AudioCNN, self).__init__()

        # Conv(7x1, 64) + BN + ReLU → Max pooling(4x1) + Dropout(0.2)
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=(7, 1), padding=(3, 0)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(4, 1)),
            nn.Dropout2d(0.2),
        )

        # Conv(10x1, 128) + BN + ReLU
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=(10, 1), padding=(4, 0)),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        # Conv(1x7, 256) + BN + ReLU
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=(1, 7), padding=(0, 3)),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )

        # Global Max Pooling + Dropout(0.5)
        self.global_pool = nn.AdaptiveMaxPool2d((1, 1))
        self.dropout = nn.Dropout(0.5)

        # Dense 128 → Softmax
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_pool(x)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


def construir_modelo():
    return AudioCNN(num_classes=10).to(device)

In [62]:
class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        w = self.X[idx]
        if w.dim() == 1:
            w = w.unsqueeze(0)
        w = w.to(device)
        if self.augment:
            w = apply_augmentation(w)
        mel = waveform_to_mel(w)
        return mel.cpu(), self.Y[idx]

In [41]:
PATIENCE_EARLY_STOP = 10
PATIENCE_REDUCE_LR  = 5
FACTOR_REDUCE_LR    = 0.5
MIN_LR              = 1e-6

In [65]:
resultados = []

for i in range(10):
    print(f"\n{'='*50}")
    print(f"  FOLD {i+1} como TEST")
    print(f"{'='*50}")

    # ── Split ────────────────────────────────────────────────────────────────
    X_test  = X_folds[i]
    Y_test  = Y_folds[i]
    X_train = np.concatenate([X_folds[j] for j in range(10) if j != i], axis=0)
    Y_train = np.concatenate([Y_folds[j] for j in range(10) if j != i], axis=0)

    # ── DataLoaders ──────────────────────────────────────────────────────────
    train_loader = torch.utils.data.DataLoader(
        AudioDataset(X_train, Y_train, augment=True),
        batch_size=32, shuffle=True,
        num_workers=0, pin_memory=True
    )
    test_loader = torch.utils.data.DataLoader(
        AudioDataset(X_test, Y_test, augment=False),
        batch_size=128, shuffle=False,
        num_workers=0, pin_memory=True
    )

    # ── Modelo ───────────────────────────────────────────────────────────────
    modelo    = construir_modelo()
    optimizer = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # ── Estado callbacks ─────────────────────────────────────────────────────
    mejor_loss        = float('inf')
    mejor_pesos       = None
    es_contador       = 0
    rlr_contador      = 0
    lr_actual         = 1e-3

    for epoch in range(100):

        # ── Entrenamiento ─────────────────────────────────────────────────────
        modelo.train()
        for X_batch, Y_batch in tqdm(train_loader, desc=f"Fold {i+1} Epoch {epoch+1:03d}", leave=False):
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(modelo(X_batch), Y_batch)
            loss.backward()
            optimizer.step()

        # ── Validación ────────────────────────────────────────────────────────
        modelo.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
                logits = modelo(X_batch)
                val_loss += criterion(logits, Y_batch).item()
                correct  += (logits.argmax(1) == Y_batch).sum().item()
                total    += Y_batch.size(0)

        val_loss /= len(test_loader)
        val_acc   = correct / total

        print(f"  Epoch {epoch+1:03d} | loss: {val_loss:.4f} | acc: {val_acc:.4f} | lr: {lr_actual:.2e}")

        # ── ReduceLROnPlateau ─────────────────────────────────────────────────
        if val_loss < mejor_loss:
            mejor_loss   = val_loss
            mejor_pesos  = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
            es_contador  = 0
            rlr_contador = 0
        else:
            es_contador  += 1
            rlr_contador += 1

            if rlr_contador >= PATIENCE_REDUCE_LR:
                lr_actual = max(lr_actual * FACTOR_REDUCE_LR, MIN_LR)
                for pg in optimizer.param_groups:
                    pg['lr'] = lr_actual
                rlr_contador = 0
                print(f"  → LR reducido a {lr_actual:.2e}")

        # ── EarlyStopping ─────────────────────────────────────────────────────
        if es_contador >= PATIENCE_EARLY_STOP:
            print(f"  → Early stopping en epoch {epoch+1}")
            break

    # ── Restaurar mejores pesos ───────────────────────────────────────────────
    modelo.load_state_dict({k: v.to(device) for k, v in mejor_pesos.items()})

    # ── Evaluación final del fold ─────────────────────────────────────────────
    modelo.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, Y_batch in test_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            correct += (modelo(X_batch).argmax(1) == Y_batch).sum().item()
            total   += Y_batch.size(0)

    acc = correct / total
    resultados.append(acc)
    print(f"\n  → Accuracy fold {i+1}: {acc:.4f}")

# ── Resultado final ───────────────────────────────────────────────────────────
print(f"\nAccuracy promedio:    {np.mean(resultados):.4f}")
print(f"Desviación estándar:  {np.std(resultados):.4f}")
print(f"Por fold: {[f'{r:.4f}' for r in resultados]}")


  FOLD 1 como TEST


  Epoch 001 | loss: 1.6645 | acc: 0.3997 | lr: 1.00e-03


  Epoch 002 | loss: 1.4995 | acc: 0.5117 | lr: 1.00e-03


  Epoch 003 | loss: 1.2600 | acc: 0.6484 | lr: 1.00e-03


  Epoch 004 | loss: 1.1744 | acc: 0.5677 | lr: 1.00e-03


...

  Epoch 017 | loss: 1.0917 | acc: 0.6771 | lr: 5.00e-04


  Epoch 018 | loss: 1.0397 | acc: 0.6927 | lr: 5.00e-04
  → LR reducido a 2.50e-04


  Epoch 019 | loss: 0.9476 | acc: 0.7630 | lr: 2.50e-04


  Epoch 020 | loss: 1.0017 | acc: 0.7448 | lr: 2.50e-04


KeyboardInterrupt: 